# Accepted Loan Final Model Comparison

Compare the selected Logistic Regression, Random Forest, HistGradientBoosting, LightGBM, XGBoost, and CatBoost models using their per-model notebook outputs.


## 1. Setup


In [1]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Cleaning").exists() and (candidate / "Modeling").exists():
            return candidate
    raise FileNotFoundError("Could not find CreditRiskRAG project root")

PROJECT_ROOT = find_project_root()
MODELING_OUTPUT_ROOT = PROJECT_ROOT / "Modeling" / "modeling_outputs"
FINAL_OUTPUT_ROOT = MODELING_OUTPUT_ROOT / "final_comparison"
TABLE_DIR = FINAL_OUTPUT_ROOT / "tables"
PLOT_DIR = FINAL_OUTPUT_ROOT / "plots"
for directory in [TABLE_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_FAMILIES = [
    "logistic_regression",
    "random_forest",
    "hist_gradient_boosting",
    "lightgbm",
    "xgboost",
    "catboost",
]
MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "hist_gradient_boosting": "HistGradientBoosting",
    "lightgbm": "LightGBM",
    "xgboost": "XGBoost",
    "catboost": "CatBoost",
}
ABLATION_MODEL_FAMILIES = MODEL_FAMILIES.copy()

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / f"final_model_{name}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def save_plot(fig, name: str) -> Path:
    path = PLOT_DIR / f"final_model_{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)
    return path

print("Project root:", PROJECT_ROOT)
print("Final comparison outputs:", FINAL_OUTPUT_ROOT)


Project root: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG
Final comparison outputs: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison


## 2. Load Per-Model Outputs


In [2]:
def model_table_path(model_family: str, table_name: str) -> Path:
    return MODELING_OUTPUT_ROOT / model_family / "tables" / f"{model_family}_{table_name}.csv"

def read_model_table(model_family: str, table_name: str) -> pd.DataFrame:
    path = model_table_path(model_family, table_name)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run the {model_family} notebook first.")
    return pd.read_csv(path)

required_tables = ["candidate_results", "selected_model_metrics", "review_volume_precision", "selected_candidate"]
available_families = []
missing_rows = []

for model_family in MODEL_FAMILIES:
    missing = [table_name for table_name in required_tables if not model_table_path(model_family, table_name).exists()]
    if missing:
        missing_rows.append({
            "model_family": model_family,
            "model_label": MODEL_LABELS[model_family],
            "missing_tables": ", ".join(missing),
            "notebook_to_run": f"Accepted_Loan_{MODEL_LABELS[model_family].replace(' ', '')}_Modeling.ipynb",
        })
    else:
        available_families.append(model_family)

missing_model_outputs = pd.DataFrame(missing_rows)
save_table(missing_model_outputs, "missing_model_outputs")
if not missing_model_outputs.empty:
    display(missing_model_outputs)

if not available_families:
    raise FileNotFoundError("No per-model outputs are available. Run at least one modeling notebook first.")

candidate_tables = []
selected_metric_tables = []
review_volume_tables = []
selected_candidate_tables = []
for model_family in available_families:
    candidate_tables.append(read_model_table(model_family, "candidate_results"))
    selected_metric_tables.append(read_model_table(model_family, "selected_model_metrics"))
    review_volume_tables.append(read_model_table(model_family, "review_volume_precision"))
    selected_candidate_tables.append(read_model_table(model_family, "selected_candidate"))

all_candidates = pd.concat(candidate_tables, ignore_index=True)
selected_metrics = pd.concat(selected_metric_tables, ignore_index=True)
review_volume_precision = pd.concat(review_volume_tables, ignore_index=True)
selected_candidates = pd.concat(selected_candidate_tables, ignore_index=True)

for frame in [all_candidates, selected_metrics, review_volume_precision, selected_candidates]:
    frame["model_label"] = frame["model_family"].map(MODEL_LABELS)

save_table(all_candidates, "candidate_summary")
save_table(selected_metrics, "selected_metrics")
save_table(review_volume_precision, "review_volume_precision")
save_table(selected_candidates, "selected_candidates")
display(selected_candidates)
display(selected_metrics)


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_missing_model_outputs.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_candidate_summary.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_selected_metrics.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_review_volume_precision.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_selected_candidates.csv


,model_family,candidate,params,fit_rows,fit_bad_rate,fit_seconds,roc_auc,pr_auc,best_f1_threshold,best_f1,best_f1_precision,best_f1_recall,target_precision_threshold,target_precision_f1,target_precision,target_precision_recall,model_label
0,logistic_regression,logistic_regression_07,"{""C"": 0.5, ""class_weight"": ""balanced"", ""l1_rat...",962641,0.1883,39.346,0.695407,0.412336,0.471801,0.470644,0.355089,0.697691,0.563511,0.447766,0.400000,0.508488,Logistic Regression
1,random_forest,random_forest_03,"{""class_weight"": ""balanced_subsample"", ""max_de...",200000,0.1883,55.434,0.694572,0.415081,0.455266,0.468385,0.359529,0.671783,0.523016,0.447541,0.400003,0.507902,Random Forest
2,hist_gradient_boosting,hist_gradient_boosting_04,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",300000,0.1883,27.427,0.700829,0.423500,0.472937,0.474259,0.359479,0.696715,0.550043,0.458965,0.400000,0.538320,HistGradientBoosting
3,lightgbm,lightgbm_06,"{""colsample_bytree"": 0.75, ""learning_rate"": 0....",300000,0.1883,8.321,0.702851,0.424770,0.459101,0.475468,0.354110,0.723382,0.547699,0.461747,0.400000,0.546038,LightGBM
4,xgboost,xgboost_05,"{""colsample_bytree"": 0.75, ""learning_rate"": 0....",300000,0.1883,11.197,0.701999,0.425400,0.470698,0.473896,0.357995,0.700770,0.546067,0.461974,0.400003,0.546667,XGBoost
5,catboost,catboost_04,"{""depth"": 5, ""iterations"": 240, ""l2_leaf_reg"":...",300000,0.1883,14.051,0.698653,0.420953,0.479729,0.472271,0.359181,0.689301,0.555200,0.453954,0.400003,0.524726,CatBoost


,model_family,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp,model_label
0,logistic_regression,logistic_regression_07,train,best_validation_f1,962641,0.188300,0.471801,0.719952,0.372912,0.214029,0.294505,0.713552,0.416930,471534,309842,51923,129342,Logistic Regression
1,logistic_regression,logistic_regression_07,train,target_validation_precision,962641,0.188300,0.563511,0.719952,0.372912,0.214029,0.345530,0.537793,0.420738,596733,184643,83782,97483,Logistic Regression
2,logistic_regression,logistic_regression_07,validation,best_validation_f1,186920,0.246763,0.471801,0.695407,0.412336,0.224051,0.355082,0.697669,0.470633,82348,58447,13945,32180,Logistic Regression
3,logistic_regression,logistic_regression_07,validation,target_validation_precision,186920,0.246763,0.563511,0.695407,0.412336,0.224051,0.400007,0.508488,0.447771,105615,35180,22671,23454,Logistic Regression
4,logistic_regression,logistic_regression_07,test,best_validation_f1,195749,0.210315,0.471801,0.700061,0.361509,0.225017,0.310541,0.709320,0.431966,89746,64834,11967,29202,Logistic Regression
5,logistic_regression,logistic_regression_07,test,target_validation_precision,195749,0.210315,0.563511,0.700061,0.361509,0.225017,0.348254,0.538026,0.422823,113127,41453,19019,22150,Logistic Regression
6,random_forest,random_forest_03,train,best_validation_f1,962641,0.188300,0.455266,0.738405,0.398883,0.194754,0.309538,0.717259,0.432450,491364,290012,51251,130014,Random Forest
7,random_forest,random_forest_03,train,target_validation_precision,962641,0.188300,0.523016,0.738405,0.398883,0.194754,0.355784,0.574358,0.439390,592863,188513,77154,104111,Random Forest
8,random_forest,random_forest_03,validation,best_validation_f1,186920,0.246763,0.455266,0.694572,0.415081,0.207616,0.359521,0.671762,0.468373,85596,55199,15140,30985,Random Forest
9,random_forest,random_forest_03,validation,target_validation_precision,186920,0.246763,0.523016,0.694572,0.415081,0.207616,0.400010,0.507902,0.447546,105656,35139,22698,23427,Random Forest


## 3. Validation And Test Comparison


In [3]:
plot_families = [family for family in MODEL_FAMILIES if family in available_families]

comparison = selected_metrics[
    selected_metrics["split"].isin(["validation", "test"])
].copy()
comparison = comparison.sort_values(
    ["operating_point", "split", "f1", "precision", "pr_auc"],
    ascending=[True, True, False, False, False],
)
save_table(comparison, "validation_test_comparison")
display(comparison)

for metric in ["f1", "precision", "recall", "pr_auc", "roc_auc", "brier_score"]:
    plot_df = comparison[comparison["operating_point"] == "best_validation_f1"]
    fig, ax = plt.subplots(figsize=(max(9, len(plot_families) * 1.35), 4.5))
    width = 0.35
    labels = [MODEL_LABELS[m] for m in plot_families]
    x = np.arange(len(labels))
    metric_by_family = plot_df.set_index(["model_family", "split"])[metric]
    val = [metric_by_family.get((m, "validation"), np.nan) for m in plot_families]
    test = [metric_by_family.get((m, "test"), np.nan) for m in plot_families]
    ax.bar(x - width/2, val, width, label="validation")
    ax.bar(x + width/2, test, width, label="test")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_title(f"Final model comparison: {metric}")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    save_plot(fig, f"{metric}_comparison")


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_validation_test_comparison.csv


,model_family,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp,model_label
22,lightgbm,lightgbm_06,test,best_validation_f1,195749,0.210315,0.459101,0.711342,0.380512,0.211718,0.316270,0.718235,0.439159,90656,63924,11600,29569,LightGBM
28,xgboost,xgboost_05,test,best_validation_f1,195749,0.210315,0.470698,0.710193,0.379590,0.212441,0.318981,0.701061,0.438462,92960,61620,12307,28862,XGBoost
16,hist_gradient_boosting,hist_gradient_boosting_04,test,best_validation_f1,195749,0.210315,0.472937,0.708493,0.377185,0.212358,0.319695,0.693143,0.437571,93856,60724,12633,28536,HistGradientBoosting
34,catboost,catboost_04,test,best_validation_f1,195749,0.210315,0.479729,0.705725,0.373955,0.216384,0.317320,0.696641,0.436029,92878,61702,12489,28680,CatBoost
10,random_forest,random_forest_03,test,best_validation_f1,195749,0.210315,0.455266,0.703753,0.371073,0.200271,0.321114,0.673832,0.434952,95931,58649,13428,27741,Random Forest
4,logistic_regression,logistic_regression_07,test,best_validation_f1,195749,0.210315,0.471801,0.700061,0.361509,0.225017,0.310541,0.709320,0.431966,89746,64834,11967,29202,Logistic Regression
20,lightgbm,lightgbm_06,validation,best_validation_f1,186920,0.246763,0.459101,0.702851,0.424770,0.217414,0.354110,0.723382,0.475468,79936,60859,12759,33366,LightGBM
14,hist_gradient_boosting,hist_gradient_boosting_04,validation,best_validation_f1,186920,0.246763,0.472937,0.700829,0.423500,0.217717,0.359479,0.696715,0.474259,83535,57260,13989,32136,HistGradientBoosting
26,xgboost,xgboost_05,validation,best_validation_f1,186920,0.246763,0.470698,0.701999,0.425400,0.217751,0.357988,0.700748,0.473884,82829,57966,13803,32322,XGBoost
32,catboost,catboost_04,validation,best_validation_f1,186920,0.246763,0.479729,0.698653,0.420953,0.219899,0.359174,0.689279,0.472260,84071,56724,14332,31793,CatBoost


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_f1_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_precision_comparison.png


Saved:

 /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_recall_comparison.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_pr_auc_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_roc_auc_comparison.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_brier_score_comparison.png


## 4. Fixed Review Volume Comparison


In [4]:
review_compare = review_volume_precision[review_volume_precision["split"].isin(["validation", "test"])].copy()
save_table(review_compare, "review_volume_comparison")
display(review_compare)

for split in ["validation", "test"]:
    plot_df = review_compare[review_compare["split"] == split]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for model_family in plot_families:
        group = plot_df[plot_df["model_family"] == model_family]
        if group.empty:
            continue
        ax.plot(group["review_pct"], group["precision"], marker="o", label=MODEL_LABELS[model_family])
    ax.set_title(f"Precision at fixed review volumes: {split}")
    ax.set_xlabel("Reviewed applications (%)")
    ax.set_ylabel("Precision")
    ax.grid(alpha=0.25)
    ax.legend()
    save_plot(fig, f"precision_by_review_volume_{split}")


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_review_volume_comparison.csv


,model_family,model,split,review_pct,review_count,captured_bad,precision,recall,base_bad_rate,lift_over_base_bad_rate,model_label
0,logistic_regression,logistic_regression_07,validation,1.0,1870,1132,0.605348,0.024542,0.246763,2.453151,Logistic Regression
1,logistic_regression,logistic_regression_07,validation,2.0,3739,2174,0.581439,0.047133,0.246763,2.356261,Logistic Regression
2,logistic_regression,logistic_regression_07,validation,5.0,9346,5073,0.542799,0.109984,0.246763,2.199675,Logistic Regression
3,logistic_regression,logistic_regression_07,validation,10.0,18692,9278,0.496362,0.201149,0.246763,2.011491,Logistic Regression
4,logistic_regression,logistic_regression_07,validation,15.0,28038,13006,0.463870,0.281973,0.246763,1.879819,Logistic Regression
...,...,...,...,...,...,...,...,...,...,...,...
91,catboost,catboost_04,test,10.0,19575,8755,0.447254,0.212660,0.210315,2.126589,CatBoost
92,catboost,catboost_04,test,15.0,29363,12253,0.417294,0.297627,0.210315,1.984135,CatBoost
93,catboost,catboost_04,test,20.0,39150,15507,0.396092,0.376667,0.210315,1.883325,CatBoost
94,catboost,catboost_04,test,25.0,48938,18457,0.377151,0.448323,0.210315,1.793264,CatBoost


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_precision_by_review_volume_validation.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_precision_by_review_volume_test.png


## 5. Grade/Subgrade Ablation

Compare the current baseline feature set against the `baseline_no_grade_subgrade` feature set for every model family with no-grade/subgrade outputs.


In [5]:
NO_GRADE_METRICS_PATH = MODELING_OUTPUT_ROOT / "tables" / "no_grade_subgrade_model_metrics.csv"
NO_GRADE_EXPORT_SUMMARY_PATH = PROJECT_ROOT / "Modeling" / "Preprocessing" / "preprocessing_outputs" / "tables" / "preprocessing_no_grade_subgrade_export_summary.csv"

if not NO_GRADE_METRICS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {NO_GRADE_METRICS_PATH}. Run the no-grade/subgrade modeling export first."
    )

no_grade_metrics = pd.read_csv(NO_GRADE_METRICS_PATH)
no_grade_metrics = no_grade_metrics[no_grade_metrics["split"].isin(["validation", "test"])].copy()

def infer_no_grade_model_family(model_name: str) -> str:
    base_name = model_name.replace("_no_grade_subgrade", "")
    for model_family in MODEL_FAMILIES:
        if base_name == model_family or base_name.startswith(f"{model_family}_"):
            return model_family
    return base_name

no_grade_metrics["model_family"] = no_grade_metrics["model"].map(infer_no_grade_model_family)
no_grade_metrics = no_grade_metrics[no_grade_metrics["model_family"].isin(ABLATION_MODEL_FAMILIES)]
no_grade_metrics["model_label"] = no_grade_metrics["model_family"].map(MODEL_LABELS)
no_grade_metrics["feature_set"] = "baseline_no_grade_subgrade"

with_grade_metrics = selected_metrics[
    (selected_metrics["model_family"].isin(ABLATION_MODEL_FAMILIES)) &
    (selected_metrics["split"].isin(["validation", "test"])) &
    (selected_metrics["operating_point"] == "best_validation_f1")
].copy()
with_grade_metrics["feature_set"] = "baseline_with_grade_subgrade"
with_grade_metrics = with_grade_metrics[[
    "model_family", "model_label", "model", "split", "feature_set",
    "roc_auc", "pr_auc", "brier_score", "precision", "recall", "f1", "threshold"
]]

no_grade_metrics = no_grade_metrics[[
    "model_family", "model_label", "model", "split", "feature_set",
    "roc_auc", "pr_auc", "brier_score", "precision", "recall", "f1", "threshold"
]]

ablation_metrics = pd.concat([with_grade_metrics, no_grade_metrics], ignore_index=True)
FEATURE_SET_LABELS = {
    "baseline_with_grade_subgrade": "with grade/subgrade",
    "baseline_no_grade_subgrade": "no grade/subgrade",
}
ablation_metrics["feature_set_label"] = ablation_metrics["feature_set"].map(FEATURE_SET_LABELS)
ablation_metrics["model_feature_label"] = ablation_metrics["model_label"] + " | " + ablation_metrics["feature_set_label"]
save_table(ablation_metrics, "grade_subgrade_ablation_metrics")
display(ablation_metrics.sort_values(["split", "model_label", "feature_set"])[[
    "model_feature_label", "model_label", "model", "split", "feature_set", "feature_set_label",
    "roc_auc", "pr_auc", "brier_score", "precision", "recall", "f1", "threshold"
]])

wide = ablation_metrics.pivot_table(
    index=["model_family", "model_label", "split"],
    columns="feature_set",
    values=["roc_auc", "pr_auc", "f1", "precision", "recall"],
    aggfunc="first",
)
wide.columns = [f"{metric}_{feature_set}" for metric, feature_set in wide.columns]
wide = wide.reset_index()

for metric in ["roc_auc", "pr_auc", "f1", "precision", "recall"]:
    wide[f"{metric}_delta"] = wide[f"{metric}_baseline_no_grade_subgrade"] - wide[f"{metric}_baseline_with_grade_subgrade"]

ablation_deltas = wide.sort_values(["split", "model_label"])
save_table(ablation_deltas, "grade_subgrade_ablation_deltas")
display(ablation_deltas)

compact_cols = [
    "model_label", "split",
    "roc_auc_baseline_with_grade_subgrade", "roc_auc_baseline_no_grade_subgrade", "roc_auc_delta",
    "pr_auc_baseline_with_grade_subgrade", "pr_auc_baseline_no_grade_subgrade", "pr_auc_delta",
    "f1_baseline_with_grade_subgrade", "f1_baseline_no_grade_subgrade", "f1_delta",
]
compact = ablation_deltas[compact_cols].copy()
compact = compact.rename(columns={"model_label": "model"})
compact["with_grade_label"] = compact["model"] + " | with grade/subgrade"
compact["no_grade_label"] = compact["model"] + " | no grade/subgrade"
save_table(compact, "grade_subgrade_ablation_compact")
display(compact[[
    "model", "split", "with_grade_label", "no_grade_label",
    "roc_auc_baseline_with_grade_subgrade", "roc_auc_baseline_no_grade_subgrade", "roc_auc_delta",
    "pr_auc_baseline_with_grade_subgrade", "pr_auc_baseline_no_grade_subgrade", "pr_auc_delta",
    "f1_baseline_with_grade_subgrade", "f1_baseline_no_grade_subgrade", "f1_delta",
]])

if NO_GRADE_EXPORT_SUMMARY_PATH.exists():
    no_grade_export_summary = pd.read_csv(NO_GRADE_EXPORT_SUMMARY_PATH)
    save_table(no_grade_export_summary, "grade_subgrade_ablation_feature_summary")
    display(no_grade_export_summary)

ablation_families = [family for family in ABLATION_MODEL_FAMILIES if family in with_grade_metrics["model_family"].unique()]
for metric in ["f1", "pr_auc", "roc_auc", "precision", "recall"]:
    plot_df = ablation_metrics[ablation_metrics["split"] == "test"].copy()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    labels = [MODEL_LABELS[m] for m in ablation_families]
    x = np.arange(len(labels))
    width = 0.35
    metric_by_family = plot_df.set_index(["model_family", "feature_set"])[metric]
    with_grade = [metric_by_family.get((m, "baseline_with_grade_subgrade"), np.nan) for m in ablation_families]
    no_grade = [metric_by_family.get((m, "baseline_no_grade_subgrade"), np.nan) for m in ablation_families]
    ax.bar(x - width/2, with_grade, width, label="with grade/subgrade")
    ax.bar(x + width/2, no_grade, width, label="without grade/subgrade")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_title(f"Grade/subgrade ablation on test: {metric}")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    save_plot(fig, f"grade_subgrade_ablation_test_{metric}")


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_grade_subgrade_ablation_metrics.csv


,model_feature_label,model_label,model,split,feature_set,feature_set_label,roc_auc,pr_auc,brier_score,precision,recall,f1,threshold
23,CatBoost | no grade/subgrade,CatBoost,catboost_03_no_grade_subgrade,test,baseline_no_grade_subgrade,no grade/subgrade,0.705990,0.374111,0.218168,0.310243,0.727683,0.435019,0.465666
11,CatBoost | with grade/subgrade,CatBoost,catboost_04,test,baseline_with_grade_subgrade,with grade/subgrade,0.705725,0.373955,0.216384,0.317320,0.696641,0.436029,0.479729
15,HistGradientBoosting | no grade/subgrade,HistGradientBoosting,hist_gradient_boosting_no_grade_subgrade,test,baseline_no_grade_subgrade,no grade/subgrade,0.709575,0.380035,0.151336,0.321367,0.688212,0.438140,0.182890
5,HistGradientBoosting | with grade/subgrade,HistGradientBoosting,hist_gradient_boosting_04,test,baseline_with_grade_subgrade,with grade/subgrade,0.708493,0.377185,0.212358,0.319695,0.693143,0.437571,0.472937
19,LightGBM | no grade/subgrade,LightGBM,lightgbm_06_no_grade_subgrade,test,baseline_no_grade_subgrade,no grade/subgrade,0.710277,0.379359,0.215386,0.317702,0.708106,0.438613,0.468473
7,LightGBM | with grade/subgrade,LightGBM,lightgbm_06,test,baseline_with_grade_subgrade,with grade/subgrade,0.711342,0.380512,0.211718,0.316270,0.718235,0.439159,0.459101
13,Logistic Regression | no grade/subgrade,Logistic Regression,logistic_regression_no_grade_subgrade,test,baseline_no_grade_subgrade,no grade/subgrade,0.697573,0.355817,0.155394,0.300132,0.750662,0.428814,0.164047
1,Logistic Regression | with grade/subgrade,Logistic Regression,logistic_regression_07,test,baseline_with_grade_subgrade,with grade/subgrade,0.700061,0.361509,0.225017,0.310541,0.709320,0.431966,0.471801
17,Random Forest | no grade/subgrade,Random Forest,random_forest_no_grade_subgrade,test,baseline_no_grade_subgrade,no grade/subgrade,0.704153,0.370329,0.203629,0.312537,0.714130,0.434789,0.443294
3,Random Forest | with grade/subgrade,Random Forest,random_forest_03,test,baseline_with_grade_subgrade,with grade/subgrade,0.703753,0.371073,0.200271,0.321114,0.673832,0.434952,0.455266


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_grade_subgrade_ablation_deltas.csv


,model_family,model_label,split,f1_baseline_no_grade_subgrade,f1_baseline_with_grade_subgrade,pr_auc_baseline_no_grade_subgrade,pr_auc_baseline_with_grade_subgrade,precision_baseline_no_grade_subgrade,precision_baseline_with_grade_subgrade,recall_baseline_no_grade_subgrade,recall_baseline_with_grade_subgrade,roc_auc_baseline_no_grade_subgrade,roc_auc_baseline_with_grade_subgrade,roc_auc_delta,pr_auc_delta,f1_delta,precision_delta,recall_delta
0,catboost,CatBoost,test,0.435019,0.436029,0.374111,0.373955,0.310243,0.317320,0.727683,0.696641,0.705990,0.705725,0.000265,0.000156,-0.001010,-0.007077,0.031042
2,hist_gradient_boosting,HistGradientBoosting,test,0.438140,0.437571,0.380035,0.377185,0.321367,0.319695,0.688212,0.693143,0.709575,0.708493,0.001082,0.002850,0.000569,0.001672,-0.004931
4,lightgbm,LightGBM,test,0.438613,0.439159,0.379359,0.380512,0.317702,0.316270,0.708106,0.718235,0.710277,0.711342,-0.001065,-0.001153,-0.000546,0.001432,-0.010129
6,logistic_regression,Logistic Regression,test,0.428814,0.431966,0.355817,0.361509,0.300132,0.310541,0.750662,0.709320,0.697573,0.700061,-0.002488,-0.005692,-0.003152,-0.010409,0.041342
8,random_forest,Random Forest,test,0.434789,0.434952,0.370329,0.371073,0.312537,0.321114,0.714130,0.673832,0.704153,0.703753,0.000400,-0.000744,-0.000163,-0.008577,0.040298
10,xgboost,XGBoost,test,0.437421,0.438462,0.379252,0.379590,0.314340,0.318981,0.718915,0.701061,0.709727,0.710193,-0.000466,-0.000338,-0.001041,-0.004641,0.017854
1,catboost,CatBoost,validation,0.472547,0.472260,0.421277,0.420953,0.352698,0.359174,0.715772,0.689279,0.699047,0.698653,0.000394,0.000324,0.000287,-0.006476,0.026493
3,hist_gradient_boosting,HistGradientBoosting,validation,0.474176,0.474259,0.425647,0.423500,0.365229,0.359479,0.675751,0.696715,0.701741,0.700829,0.000912,0.002147,-0.000083,0.005750,-0.020964
5,lightgbm,LightGBM,validation,0.475030,0.475468,0.424132,0.424770,0.357799,0.354110,0.706515,0.723382,0.702179,0.702851,-0.000672,-0.000638,-0.000438,0.003689,-0.016867
7,logistic_regression,Logistic Regression,validation,0.468918,0.470633,0.408532,0.412336,0.344783,0.355082,0.732726,0.697669,0.692454,0.695407,-0.002953,-0.003804,-0.001715,-0.010299,0.035057


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_grade_subgrade_ablation_compact.csv


...

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_f1.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_pr_auc.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_roc_auc.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_precision.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/plots/final_model_grade_subgrade_ablation_test_recall.png


## 6. Recommendation


In [6]:
preferred_operating_point = "best_validation_f1"

validation_rank = (
    selected_metrics[
        (selected_metrics["split"] == "validation") &
        (selected_metrics["operating_point"] == preferred_operating_point)
    ]
    .sort_values(["f1", "precision", "pr_auc", "roc_auc"], ascending=False)
    .reset_index(drop=True)
)

preferred_validation = validation_rank.iloc[0]
preferred_model_family = preferred_validation["model_family"]
preferred_candidate = preferred_validation["model"]
preferred_feature_set = "baseline_with_grade_subgrade"

preferred_test = selected_metrics[
    (selected_metrics["model_family"] == preferred_model_family) &
    (selected_metrics["model"] == preferred_candidate) &
    (selected_metrics["split"] == "test") &
    (selected_metrics["operating_point"] == preferred_operating_point)
].iloc[0]

winner_confusion_matrix = read_model_table(preferred_model_family, "confusion_matrix")
recommended_predicted_label_counts = (
    winner_confusion_matrix[
        (winner_confusion_matrix["candidate"] == preferred_candidate) &
        (winner_confusion_matrix["split"].isin(["validation", "test"])) &
        (winner_confusion_matrix["operating_point"] == preferred_operating_point)
    ]
    .groupby([
        "model_family", "candidate", "split", "operating_point", "threshold",
        "predicted_label", "predicted_class"
    ], as_index=False)["count"]
    .sum()
    .rename(columns={"count": "predicted_count"})
    .assign(split_order=lambda df: df["split"].map({"validation": 0, "test": 1}))
    .sort_values(["split_order", "predicted_label"])
    .drop(columns="split_order")
)
recommended_predicted_label_counts["predicted_share"] = (
    recommended_predicted_label_counts["predicted_count"] /
    recommended_predicted_label_counts.groupby("split")["predicted_count"].transform("sum")
).round(6)

def predicted_value(split: str, predicted_label: int, column: str) -> float:
    match = recommended_predicted_label_counts[
        (recommended_predicted_label_counts["split"] == split) &
        (recommended_predicted_label_counts["predicted_label"] == predicted_label)
    ]
    if match.empty:
        return np.nan
    value = match.iloc[0][column]
    return int(value) if column == "predicted_count" else float(value)

recommendation = pd.DataFrame([{
    "recommended_model_family": preferred_model_family,
    "recommended_model_label": MODEL_LABELS[preferred_model_family],
    "recommended_candidate": preferred_candidate,
    "recommended_feature_set": preferred_feature_set,
    "recommended_operating_point": preferred_operating_point,
    "selection_basis": (
        "Select the available model with the highest validation F1 at the best-validation-F1 "
        "operating point, using validation precision, PR-AUC, and ROC-AUC as tie breakers. "
        "Test metrics are reported after selection and are not used to choose the winner."
    ),
    "validation_f1": preferred_validation["f1"],
    "validation_precision": preferred_validation["precision"],
    "validation_recall": preferred_validation["recall"],
    "validation_pr_auc": preferred_validation["pr_auc"],
    "validation_roc_auc": preferred_validation["roc_auc"],
    "validation_predicted_0_count": predicted_value("validation", 0, "predicted_count"),
    "validation_predicted_0_share": predicted_value("validation", 0, "predicted_share"),
    "validation_predicted_1_count": predicted_value("validation", 1, "predicted_count"),
    "validation_predicted_1_share": predicted_value("validation", 1, "predicted_share"),
    "test_f1": preferred_test["f1"],
    "test_precision": preferred_test["precision"],
    "test_recall": preferred_test["recall"],
    "test_pr_auc": preferred_test["pr_auc"],
    "test_roc_auc": preferred_test["roc_auc"],
    "test_predicted_0_count": predicted_value("test", 0, "predicted_count"),
    "test_predicted_0_share": predicted_value("test", 0, "predicted_share"),
    "test_predicted_1_count": predicted_value("test", 1, "predicted_count"),
    "test_predicted_1_share": predicted_value("test", 1, "predicted_share"),
}])

save_table(validation_rank, "ranking_by_validation_f1")
save_table(recommended_predicted_label_counts, "predicted_label_counts")
save_table(recommendation, "recommendation")
display(validation_rank)
display(recommended_predicted_label_counts)
display(recommendation)

feature_set_rank = (
    ablation_metrics[
        (ablation_metrics["split"] == "validation")
        & (ablation_metrics["feature_set"].isin(["baseline_with_grade_subgrade", "baseline_no_grade_subgrade"]))
    ]
    .sort_values(["f1", "precision", "pr_auc", "roc_auc"], ascending=False)
    .reset_index(drop=True)
)
feature_set_rank["selection_rank"] = feature_set_rank.index + 1
feature_set_rank = feature_set_rank[[
    "selection_rank", "model_feature_label", "model_label", "model", "feature_set",
    "feature_set_label", "split", "f1", "precision", "recall", "pr_auc", "roc_auc"
]]
save_table(feature_set_rank, "ranking_by_validation_f1_with_feature_set")
display(feature_set_rank)

recommended_feature_set_row = feature_set_rank.iloc[0]
recommendation_with_feature_set = recommendation.copy()
recommendation_with_feature_set["recommended_model_feature_label"] = recommended_feature_set_row["model_feature_label"]
recommendation_with_feature_set["recommended_candidate_with_feature_set"] = recommended_feature_set_row["model"]
recommendation_with_feature_set["recommended_feature_set"] = recommended_feature_set_row["feature_set"]
recommendation_with_feature_set["recommended_feature_set_label"] = recommended_feature_set_row["feature_set_label"]
recommendation_with_feature_set["recommended_validation_f1_with_feature_set_rank"] = recommended_feature_set_row["f1"]
save_table(recommendation_with_feature_set, "recommendation_with_feature_set_label")
display(recommendation_with_feature_set[[
    "recommended_model_feature_label", "recommended_model_label", "recommended_candidate_with_feature_set",
    "recommended_feature_set", "recommended_feature_set_label", "recommended_operating_point",
    "recommended_validation_f1_with_feature_set_rank", "validation_f1", "test_f1",
]])

per_class_tables = []
for model_family in available_families:
    per_class_tables.append(read_model_table(model_family, "per_class_metrics"))
all_model_per_class_metrics = pd.concat(per_class_tables, ignore_index=True)
all_model_per_class_metrics["model_label"] = all_model_per_class_metrics["model_family"].map(MODEL_LABELS)
save_table(all_model_per_class_metrics, "per_class_metrics")
display(all_model_per_class_metrics)


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_ranking_by_validation_f1.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_predicted_label_counts.csv


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_recommendation.csv


,model_family,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp,model_label
0,lightgbm,lightgbm_06,validation,best_validation_f1,186920,0.246763,0.459101,0.702851,0.424770,0.217414,0.354110,0.723382,0.475468,79936,60859,12759,33366,LightGBM
1,hist_gradient_boosting,hist_gradient_boosting_04,validation,best_validation_f1,186920,0.246763,0.472937,0.700829,0.423500,0.217717,0.359479,0.696715,0.474259,83535,57260,13989,32136,HistGradientBoosting
2,xgboost,xgboost_05,validation,best_validation_f1,186920,0.246763,0.470698,0.701999,0.425400,0.217751,0.357988,0.700748,0.473884,82829,57966,13803,32322,XGBoost
3,catboost,catboost_04,validation,best_validation_f1,186920,0.246763,0.479729,0.698653,0.420953,0.219899,0.359174,0.689279,0.472260,84071,56724,14332,31793,CatBoost
4,logistic_regression,logistic_regression_07,validation,best_validation_f1,186920,0.246763,0.471801,0.695407,0.412336,0.224051,0.355082,0.697669,0.470633,82348,58447,13945,32180,Logistic Regression
5,random_forest,random_forest_03,validation,best_validation_f1,186920,0.246763,0.455266,0.694572,0.415081,0.207616,0.359521,0.671762,0.468373,85596,55199,15140,30985,Random Forest


,model_family,candidate,split,operating_point,threshold,predicted_label,predicted_class,predicted_count,predicted_share
2,lightgbm,lightgbm_06,validation,best_validation_f1,0.459101,0,Fully Paid,92695,0.495907
3,lightgbm,lightgbm_06,validation,best_validation_f1,0.459101,1,Charged Off,94225,0.504093
0,lightgbm,lightgbm_06,test,best_validation_f1,0.459101,0,Fully Paid,102256,0.522383
1,lightgbm,lightgbm_06,test,best_validation_f1,0.459101,1,Charged Off,93493,0.477617


,recommended_model_family,recommended_model_label,recommended_candidate,recommended_feature_set,recommended_operating_point,selection_basis,validation_f1,validation_precision,validation_recall,validation_pr_auc,...,validation_predicted_1_share,test_f1,test_precision,test_recall,test_pr_auc,test_roc_auc,test_predicted_0_count,test_predicted_0_share,test_predicted_1_count,test_predicted_1_share
0,lightgbm,LightGBM,lightgbm_06,baseline_with_grade_subgrade,best_validation_f1,Select the available model with the highest va...,0.475468,0.35411,0.723382,0.42477,...,0.504093,0.439159,0.31627,0.718235,0.380512,0.711342,102256,0.522383,93493,0.477617


...

,selection_rank,model_feature_label,model_label,model,feature_set,feature_set_label,split,f1,precision,recall,pr_auc,roc_auc
0,1,LightGBM | with grade/subgrade,LightGBM,lightgbm_06,baseline_with_grade_subgrade,with grade/subgrade,validation,0.475468,0.354110,0.723382,0.424770,0.702851
1,2,LightGBM | no grade/subgrade,LightGBM,lightgbm_06_no_grade_subgrade,baseline_no_grade_subgrade,no grade/subgrade,validation,0.475030,0.357799,0.706515,0.424132,0.702179
2,3,HistGradientBoosting | with grade/subgrade,HistGradientBoosting,hist_gradient_boosting_04,baseline_with_grade_subgrade,with grade/subgrade,validation,0.474259,0.359479,0.696715,0.423500,0.700829
3,4,HistGradientBoosting | no grade/subgrade,HistGradientBoosting,hist_gradient_boosting_no_grade_subgrade,baseline_no_grade_subgrade,no grade/subgrade,validation,0.474176,0.365229,0.675751,0.425647,0.701741
4,5,XGBoost | no grade/subgrade,XGBoost,xgboost_05_no_grade_subgrade,baseline_no_grade_subgrade,no grade/subgrade,validation,0.474049,0.354651,0.714645,0.425118,0.701651
5,6,XGBoost | with grade/subgrade,XGBoost,xgboost_05,baseline_with_grade_subgrade,with grade/subgrade,validation,0.473884,0.357988,0.700748,0.425400,0.701999
6,7,CatBoost | no grade/subgrade,CatBoost,catboost_03_no_grade_subgrade,baseline_no_grade_subgrade,no grade/subgrade,validation,0.472547,0.352698,0.715772,0.421277,0.699047
7,8,CatBoost | with grade/subgrade,CatBoost,catboost_04,baseline_with_grade_subgrade,with grade/subgrade,validation,0.472260,0.359174,0.689279,0.420953,0.698653
8,9,Random Forest | no grade/subgrade,Random Forest,random_forest_no_grade_subgrade,baseline_no_grade_subgrade,no grade/subgrade,validation,0.471600,0.351879,0.714797,0.416275,0.696983
9,10,Logistic Regression | with grade/subgrade,Logistic Regression,logistic_regression_07,baseline_with_grade_subgrade,with grade/subgrade,validation,0.470633,0.355082,0.697669,0.412336,0.695407


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_recommendation_with_feature_set_label.csv


,recommended_model_feature_label,recommended_model_label,recommended_candidate_with_feature_set,recommended_feature_set,recommended_feature_set_label,recommended_operating_point,recommended_validation_f1_with_feature_set_rank,validation_f1,test_f1
0,LightGBM | with grade/subgrade,LightGBM,lightgbm_06,baseline_with_grade_subgrade,with grade/subgrade,best_validation_f1,0.475468,0.475468,0.439159


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_per_class_metrics.csv


,model_family,candidate,split,operating_point,threshold,class_label,class_name,precision,recall,f1,support,model_label
0,logistic_regression,logistic_regression_07,train,best_validation_f1,0.471801,0,Fully Paid,0.900808,0.603466,0.722750,781376,Logistic Regression
1,logistic_regression,logistic_regression_07,train,best_validation_f1,0.471801,1,Charged Off,0.294505,0.713552,0.416930,181265,Logistic Regression
2,logistic_regression,logistic_regression_07,train,target_validation_precision,0.563511,0,Fully Paid,0.876884,0.763695,0.816385,781376,Logistic Regression
3,logistic_regression,logistic_regression_07,train,target_validation_precision,0.563511,1,Charged Off,0.345530,0.537793,0.420738,181265,Logistic Regression
4,logistic_regression,logistic_regression_07,validation,best_validation_f1,0.471801,0,Fully Paid,0.855182,0.584879,0.694662,140795,Logistic Regression
...,...,...,...,...,...,...,...,...,...,...,...,...
67,catboost,catboost_04,validation,target_validation_precision,0.555200,1,Charged Off,0.400003,0.524726,0.453954,46125,CatBoost
68,catboost,catboost_04,test,best_validation_f1,0.479729,0,Fully Paid,0.881471,0.600841,0.714592,154580,CatBoost
69,catboost,catboost_04,test,best_validation_f1,0.479729,1,Charged Off,0.317320,0.696641,0.436029,41169,CatBoost
70,catboost,catboost_04,test,target_validation_precision,0.555200,0,Fully Paid,0.857925,0.737844,0.793367,154580,CatBoost


## 7. PR-AUC Optimized Advanced Models

Compare the separate PR-AUC-monitored LightGBM, XGBoost, and CatBoost experiment against the current F1-selected recommendation.


In [7]:
PR_AUC_OUTPUT_ROOT = MODELING_OUTPUT_ROOT / "pr_auc_optimized" / "tables"
PR_AUC_RANKING_PATH = PR_AUC_OUTPUT_ROOT / "pr_auc_optimized_ranking_by_validation_pr_auc.csv"
PR_AUC_TEST_RANKING_PATH = PR_AUC_OUTPUT_ROOT / "pr_auc_optimized_test_ranking.csv"
PR_AUC_RECOMMENDATION_PATH = PR_AUC_OUTPUT_ROOT / "pr_auc_optimized_recommendation.csv"

missing_pr_auc_outputs = [
    path for path in [PR_AUC_RANKING_PATH, PR_AUC_TEST_RANKING_PATH, PR_AUC_RECOMMENDATION_PATH]
    if not path.exists()
]
if missing_pr_auc_outputs:
    raise FileNotFoundError(
        "Missing PR-AUC optimized outputs. Run scripts/run_pr_auc_optimized_advanced_models.py first: "
        + ", ".join(str(path) for path in missing_pr_auc_outputs)
    )

pr_auc_validation_rank = pd.read_csv(PR_AUC_RANKING_PATH)
pr_auc_test_rank = pd.read_csv(PR_AUC_TEST_RANKING_PATH)
pr_auc_recommendation = pd.read_csv(PR_AUC_RECOMMENDATION_PATH)

save_table(pr_auc_validation_rank, "pr_auc_optimized_ranking_by_validation_pr_auc")
save_table(pr_auc_test_rank, "pr_auc_optimized_test_ranking")
save_table(pr_auc_recommendation, "pr_auc_optimized_recommendation")

current_rec = recommendation_with_feature_set.iloc[0]
pr_auc_rec = pr_auc_recommendation.iloc[0]

selection_strategy_comparison = pd.DataFrame([
    {
        "selection_strategy": "F1-selected current final comparison",
        "recommended_model_feature_label": current_rec["recommended_model_feature_label"],
        "recommended_candidate": current_rec["recommended_candidate_with_feature_set"],
        "recommended_feature_set": current_rec["recommended_feature_set"],
        "validation_pr_auc": current_rec["validation_pr_auc"],
        "validation_f1": current_rec["validation_f1"],
        "validation_precision": current_rec["validation_precision"],
        "validation_recall": current_rec["validation_recall"],
        "validation_predicted_reject_share": current_rec["validation_predicted_1_share"],
        "test_pr_auc": current_rec["test_pr_auc"],
        "test_f1": current_rec["test_f1"],
        "test_precision": current_rec["test_precision"],
        "test_recall": current_rec["test_recall"],
        "test_predicted_reject_share": current_rec["test_predicted_1_share"],
    },
    {
        "selection_strategy": "PR-AUC-selected advanced experiment",
        "recommended_model_feature_label": pr_auc_rec["recommended_model_feature_label"],
        "recommended_candidate": pr_auc_rec["recommended_candidate"],
        "recommended_feature_set": pr_auc_rec["recommended_feature_set"],
        "validation_pr_auc": pr_auc_rec["validation_pr_auc"],
        "validation_f1": pr_auc_rec["validation_f1"],
        "validation_precision": pr_auc_rec["validation_precision"],
        "validation_recall": pr_auc_rec["validation_recall"],
        "validation_predicted_reject_share": pr_auc_rec["validation_predicted_reject_share"],
        "test_pr_auc": pr_auc_rec["test_pr_auc"],
        "test_f1": pr_auc_rec["test_f1"],
        "test_precision": pr_auc_rec["test_precision"],
        "test_recall": pr_auc_rec["test_recall"],
        "test_predicted_reject_share": pr_auc_rec["test_predicted_reject_share"],
    },
])
save_table(selection_strategy_comparison, "selection_strategy_comparison")

display(pr_auc_validation_rank[[
    "selection_rank", "model_feature_label", "model", "feature_set_label",
    "pr_auc", "roc_auc", "f1", "precision", "recall",
    "predicted_reject_share", "false_rejection_share_among_rejects"
]])
display(selection_strategy_comparison)


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_pr_auc_optimized_ranking_by_validation_pr_auc.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_pr_auc_optimized_test_ranking.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_pr_auc_optimized_recommendation.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_selection_strategy_comparison.csv


,selection_rank,model_feature_label,model,feature_set_label,pr_auc,roc_auc,f1,precision,recall,predicted_reject_share,false_rejection_share_among_rejects
0,1,XGBoost | with grade/subgrade,xgboost_pr_auc_05_with_grade_subgrade,with grade/subgrade,0.425400,0.701999,0.473896,0.357995,0.700770,0.483036,0.642005
1,2,XGBoost | no grade/subgrade,xgboost_pr_auc_05_no_grade_subgrade,no grade/subgrade,0.425118,0.701651,0.474057,0.354654,0.714667,0.497256,0.645346
2,3,LightGBM | no grade/subgrade,lightgbm_pr_auc_05_no_grade_subgrade,no grade/subgrade,0.424844,0.702350,0.474987,0.353654,0.723057,0.504515,0.646346
3,4,LightGBM | with grade/subgrade,lightgbm_pr_auc_06_with_grade_subgrade,with grade/subgrade,0.424770,0.702851,0.475458,0.354103,0.723360,0.504087,0.645897
4,5,CatBoost | no grade/subgrade,catboost_pr_auc_03_no_grade_subgrade,no grade/subgrade,0.421277,0.699047,0.472547,0.352698,0.715772,0.500786,0.647302
5,6,CatBoost | with grade/subgrade,catboost_pr_auc_03_with_grade_subgrade,with grade/subgrade,0.421236,0.699036,0.471924,0.354769,0.704607,0.490097,0.645231


,selection_strategy,recommended_model_feature_label,recommended_candidate,recommended_feature_set,validation_pr_auc,validation_f1,validation_precision,validation_recall,validation_predicted_reject_share,test_pr_auc,test_f1,test_precision,test_recall,test_predicted_reject_share
0,F1-selected current final comparison,LightGBM | with grade/subgrade,lightgbm_06,baseline_with_grade_subgrade,0.42477,0.475468,0.354110,0.723382,0.504093,0.380512,0.439159,0.316270,0.718235,0.477617
1,PR-AUC-selected advanced experiment,XGBoost | with grade/subgrade,xgboost_pr_auc_05_with_grade_subgrade,baseline_with_grade_subgrade,0.42540,0.473896,0.357995,0.700770,0.483036,0.379590,0.438462,0.318981,0.701061,0.462235


## 8. Missingness Challenger Dataset Comparison

Compare all six model families across baseline with grade/subgrade, baseline no grade/subgrade, and missingness challenger datasets.


In [8]:
MISSINGNESS_OUTPUT_ROOT = MODELING_OUTPUT_ROOT / "missingness_challenger" / "tables"
MISSINGNESS_COMPARISON_PATH = MISSINGNESS_OUTPUT_ROOT / "missingness_challenger_validation_test_dataset_comparison.csv"
MISSINGNESS_RANKING_PATH = MISSINGNESS_OUTPUT_ROOT / "missingness_challenger_ranking_by_validation_f1.csv"
MISSINGNESS_BEST_BY_MODEL_PATH = MISSINGNESS_OUTPUT_ROOT / "missingness_challenger_best_dataset_by_model_family.csv"
MISSINGNESS_RECOMMENDATION_PATH = MISSINGNESS_OUTPUT_ROOT / "missingness_challenger_recommendation.csv"
MISSINGNESS_NO_GRADE_OUTPUT_ROOT = MODELING_OUTPUT_ROOT / "missingness_challenger_no_grade_subgrade" / "tables"
MISSINGNESS_NO_GRADE_COMPARISON_PATH = MISSINGNESS_NO_GRADE_OUTPUT_ROOT / "missingness_challenger_no_grade_subgrade_validation_test_comparison.csv"

missing_missingness_outputs = [
    path for path in [
        MISSINGNESS_COMPARISON_PATH,
        MISSINGNESS_RANKING_PATH,
        MISSINGNESS_BEST_BY_MODEL_PATH,
        MISSINGNESS_RECOMMENDATION_PATH,
        MISSINGNESS_NO_GRADE_COMPARISON_PATH,
    ]
    if not path.exists()
]
if missing_missingness_outputs:
    raise FileNotFoundError(
        "Missing missingness challenger outputs. Run scripts/run_missingness_challenger_all_models.py first: "
        + ", ".join(str(path) for path in missing_missingness_outputs)
    )

missingness_dataset_comparison = pd.read_csv(MISSINGNESS_COMPARISON_PATH)
missingness_no_grade_comparison = pd.read_csv(MISSINGNESS_NO_GRADE_COMPARISON_PATH)
missingness_dataset_comparison = pd.concat(
    [missingness_dataset_comparison, missingness_no_grade_comparison],
    ignore_index=True,
    sort=False,
)
missingness_dataset_comparison = missingness_dataset_comparison.drop_duplicates(
    subset=["model_family", "model", "dataset", "split", "operating_point"],
    keep="last",
)

missingness_validation_rank = (
    missingness_dataset_comparison[
        (missingness_dataset_comparison["split"] == "validation")
        & (missingness_dataset_comparison["operating_point"] == "best_validation_f1")
    ]
    .sort_values(["f1", "precision", "pr_auc"], ascending=False)
    .reset_index(drop=True)
)
missingness_validation_rank["selection_rank"] = missingness_validation_rank.index + 1

missingness_validation_pr_auc_rank = (
    missingness_dataset_comparison[
        (missingness_dataset_comparison["split"] == "validation")
        & (missingness_dataset_comparison["operating_point"] == "best_validation_f1")
    ]
    .sort_values(["pr_auc", "roc_auc", "f1"], ascending=False)
    .reset_index(drop=True)
)
missingness_validation_pr_auc_rank["selection_rank"] = missingness_validation_pr_auc_rank.index + 1

missingness_best_by_model = (
    missingness_validation_rank
    .sort_values(["model_family", "selection_rank"])
    .groupby("model_family", as_index=False)
    .first()
)
missingness_recommendation = missingness_validation_rank.iloc[[0]].copy()
missingness_pr_auc_recommendation = missingness_validation_pr_auc_rank.iloc[[0]].copy()

save_table(missingness_dataset_comparison, "missingness_challenger_dataset_comparison")
save_table(missingness_validation_rank, "missingness_challenger_ranking_by_validation_f1")
save_table(missingness_validation_pr_auc_rank, "missingness_challenger_ranking_by_validation_pr_auc")
save_table(missingness_best_by_model, "missingness_challenger_best_dataset_by_model_family")
save_table(missingness_recommendation, "missingness_challenger_recommendation")
save_table(missingness_pr_auc_recommendation, "missingness_challenger_pr_auc_recommendation")

display(missingness_validation_rank[[
    "selection_rank", "model_dataset_label", "model", "dataset_label",
    "f1", "precision", "recall", "pr_auc", "roc_auc", "predicted_reject_share"
]].head(18))

display(missingness_best_by_model[[
    "model_label", "dataset_label", "model", "f1", "precision", "recall",
    "pr_auc", "roc_auc", "predicted_reject_share"
]])

display(missingness_recommendation[[
    "model_dataset_label", "model", "dataset_label", "f1", "precision",
    "recall", "pr_auc", "roc_auc", "predicted_reject_share"
]])

display(missingness_validation_pr_auc_rank[[
    "selection_rank", "model_dataset_label", "model", "dataset_label",
    "pr_auc", "roc_auc", "f1", "precision", "recall", "predicted_reject_share"
]].head(18))

display(missingness_pr_auc_recommendation[[
    "model_dataset_label", "model", "dataset_label", "pr_auc", "roc_auc",
    "f1", "precision", "recall", "predicted_reject_share"
]])


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_missingness_challenger_dataset_comparison.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_missingness_challenger_ranking_by_validation_f1.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_missingness_challenger_ranking_by_validation_pr_auc.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_missingness_challenger_best_dataset_by_model_family.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modelin

,selection_rank,model_dataset_label,model,dataset_label,f1,precision,recall,pr_auc,roc_auc,predicted_reject_share
0,1,LightGBM | baseline with grade/subgrade,lightgbm_06,baseline with grade/subgrade,0.475468,0.354110,0.723382,0.424770,0.702851,0.504093
1,2,LightGBM | missingness challenger no grade/sub...,lightgbm_06_missingness_challenger_no_grade_su...,missingness challenger no grade/subgrade,0.475249,0.355631,0.716119,0.424751,0.702609,0.496897
2,3,LightGBM | missingness challenger,lightgbm_03_missingness_challenger,missingness challenger,0.475150,0.361194,0.694157,0.424804,0.702981,0.474240
3,4,LightGBM | baseline no grade/subgrade,lightgbm_06_no_grade_subgrade,baseline no grade/subgrade,0.475030,0.357799,0.706515,0.424132,0.702179,0.487262
4,5,HistGradientBoosting | baseline with grade/sub...,hist_gradient_boosting_04,baseline with grade/subgrade,0.474259,0.359479,0.696715,0.423500,0.700829,0.478258
5,6,HistGradientBoosting | baseline no grade/subgrade,hist_gradient_boosting_no_grade_subgrade,baseline no grade/subgrade,0.474176,0.365229,0.675751,0.425647,0.701741,0.456564
6,7,XGBoost | missingness challenger no grade/subg...,xgboost_05_missingness_challenger_no_grade_sub...,missingness challenger no grade/subgrade,0.474124,0.355525,0.711458,0.425318,0.701878,0.493810
7,8,XGBoost | missingness challenger,xgboost_05_missingness_challenger,missingness challenger,0.474077,0.354607,0.714949,0.425707,0.702088,0.497518
8,9,XGBoost | baseline no grade/subgrade,xgboost_05_no_grade_subgrade,baseline no grade/subgrade,0.474049,0.354651,0.714645,0.425118,0.701651,0.497245
9,10,HistGradientBoosting | missingness challenger,hist_gradient_boosting_06_missingness_challenger,missingness challenger,0.473904,0.353878,0.717138,0.423860,0.700996,0.500070


,model_label,dataset_label,model,f1,precision,recall,pr_auc,roc_auc,predicted_reject_share
0,CatBoost,missingness challenger no grade/subgrade,catboost_02_missingness_challenger_no_grade_su...,0.472597,0.354985,0.706753,0.419466,0.698011,0.491290
1,HistGradientBoosting,baseline with grade/subgrade,hist_gradient_boosting_04,0.474259,0.359479,0.696715,0.423500,0.700829,0.478258
2,LightGBM,baseline with grade/subgrade,lightgbm_06,0.475468,0.354110,0.723382,0.424770,0.702851,0.504093
3,Logistic Regression,missingness challenger no grade/subgrade,logistic_regression_08_missingness_challenger_...,0.471085,0.354774,0.700856,0.412345,0.695499,0.487481
4,Random Forest,missingness challenger no grade/subgrade,random_forest_02_missingness_challenger_no_gra...,0.471667,0.348909,0.727696,0.416505,0.696912,0.514659
5,XGBoost,missingness challenger no grade/subgrade,xgboost_05_missingness_challenger_no_grade_sub...,0.474124,0.355525,0.711458,0.425318,0.701878,0.493810


,model_dataset_label,model,dataset_label,f1,precision,recall,pr_auc,roc_auc,predicted_reject_share
0,LightGBM | baseline with grade/subgrade,lightgbm_06,baseline with grade/subgrade,0.475468,0.35411,0.723382,0.42477,0.702851,0.504093


,selection_rank,model_dataset_label,model,dataset_label,pr_auc,roc_auc,f1,precision,recall,predicted_reject_share
0,1,XGBoost | missingness challenger,xgboost_05_missingness_challenger,missingness challenger,0.425707,0.702088,0.474077,0.354607,0.714949,0.497518
1,2,HistGradientBoosting | baseline no grade/subgrade,hist_gradient_boosting_no_grade_subgrade,baseline no grade/subgrade,0.425647,0.701741,0.474176,0.365229,0.675751,0.456564
2,3,XGBoost | baseline with grade/subgrade,xgboost_05,baseline with grade/subgrade,0.425400,0.701999,0.473884,0.357988,0.700748,0.483030
3,4,XGBoost | missingness challenger no grade/subg...,xgboost_05_missingness_challenger_no_grade_sub...,missingness challenger no grade/subgrade,0.425318,0.701878,0.474124,0.355525,0.711458,0.493810
4,5,XGBoost | baseline no grade/subgrade,xgboost_05_no_grade_subgrade,baseline no grade/subgrade,0.425118,0.701651,0.474049,0.354651,0.714645,0.497245
5,6,LightGBM | missingness challenger,lightgbm_03_missingness_challenger,missingness challenger,0.424804,0.702981,0.475150,0.361194,0.694157,0.474240
6,7,LightGBM | baseline with grade/subgrade,lightgbm_06,baseline with grade/subgrade,0.424770,0.702851,0.475468,0.354110,0.723382,0.504093
7,8,LightGBM | missingness challenger no grade/sub...,lightgbm_06_missingness_challenger_no_grade_su...,missingness challenger no grade/subgrade,0.424751,0.702609,0.475249,0.355631,0.716119,0.496897
8,9,LightGBM | baseline no grade/subgrade,lightgbm_06_no_grade_subgrade,baseline no grade/subgrade,0.424132,0.702179,0.475030,0.357799,0.706515,0.487262
9,10,HistGradientBoosting | missingness challenger,hist_gradient_boosting_06_missingness_challenger,missingness challenger,0.423860,0.700996,0.473904,0.353878,0.717138,0.500070


,model_dataset_label,model,dataset_label,pr_auc,roc_auc,f1,precision,recall,predicted_reject_share
0,XGBoost | missingness challenger,xgboost_05_missingness_challenger,missingness challenger,0.425707,0.702088,0.474077,0.354607,0.714949,0.497518


## 9. Operating Policy Warning

PR-AUC ranks models without choosing an action threshold. Reject-share values from the best-F1 threshold should not be interpreted as an automatic rejection policy.


In [9]:
POLICY_WARNING_PATH = TABLE_DIR / "final_model_pr_auc_ranking_with_f1_threshold_warning.csv"
FIXED_POLICY_PATH = TABLE_DIR / "final_model_top_pr_auc_fixed_review_volume_policy.csv"
FIXED_POLICY_MISSING_PATH = TABLE_DIR / "final_model_top_pr_auc_fixed_review_volume_missing.csv"
ECONOMIC_SELECTED_PATH = TABLE_DIR / "final_model_economic_underwriting_selected_thresholds.csv"
ECONOMIC_APPLICATION_PATH = TABLE_DIR / "final_model_economic_underwriting_policy_application.csv"
ECONOMIC_RECOMMENDATION_PATH = TABLE_DIR / "final_model_economic_underwriting_recommendation.csv"

OPERATING_POLICY_SCRIPT = PROJECT_ROOT / "scripts" / "create_operating_policy_analysis.py"
if not OPERATING_POLICY_SCRIPT.exists():
    raise FileNotFoundError(OPERATING_POLICY_SCRIPT)

completed = subprocess.run(
    [sys.executable, str(OPERATING_POLICY_SCRIPT)],
    cwd=str(PROJECT_ROOT),
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

missing_policy_outputs = [
    path for path in [
        POLICY_WARNING_PATH,
        FIXED_POLICY_PATH,
        FIXED_POLICY_MISSING_PATH,
        ECONOMIC_SELECTED_PATH,
        ECONOMIC_APPLICATION_PATH,
        ECONOMIC_RECOMMENDATION_PATH,
    ]
    if not path.exists()
]
if missing_policy_outputs:
    raise FileNotFoundError(
        "Missing operating policy outputs. Run scripts/create_operating_policy_analysis.py and "
        "scripts/create_economic_underwriting_policy.py first: "
        + ", ".join(str(path) for path in missing_policy_outputs)
    )

pr_auc_ranking_with_warning = pd.read_csv(POLICY_WARNING_PATH)
fixed_review_policy = pd.read_csv(FIXED_POLICY_PATH)
fixed_review_policy_missing = pd.read_csv(FIXED_POLICY_MISSING_PATH)
economic_selected = pd.read_csv(ECONOMIC_SELECTED_PATH)
economic_application = pd.read_csv(ECONOMIC_APPLICATION_PATH)
economic_recommendation = pd.read_csv(ECONOMIC_RECOMMENDATION_PATH)

display(pr_auc_ranking_with_warning[[
    "selection_rank", "model_dataset_label", "split", "pr_auc", "roc_auc",
    "f1_threshold_f1", "f1_threshold_precision", "f1_threshold_recall",
    "f1_threshold_predicted_reject_share", "operating_policy_warning"
]].head(12))

display(fixed_review_policy[[
    "selection_rank", "model_dataset_label", "split", "review_pct", "business_policy",
    "review_count", "captured_bad", "precision", "recall", "base_bad_rate",
    "lift_over_base_bad_rate", "avoided_auto_reject_share_vs_f1_threshold"
]].head(48))

if not fixed_review_policy_missing.empty:
    display(fixed_review_policy_missing)

display(economic_selected[[
    "policy_type", "calibration_method", "threshold", "predicted_reject_share",
    "precision_bad_rate_among_rejected", "recall_default_capture",
    "total_portfolio_value", "value_per_applicant",
    "default_loss_saved", "good_borrower_opportunity_cost",
    "economic_break_even_probability", "max_reject_share_constraint"
]])

display(economic_application[[
    "policy_type", "calibration_method", "split", "threshold",
    "predicted_reject_share", "approved_share", "precision_bad_rate_among_rejected",
    "recall_default_capture", "approved_bad_rate", "total_portfolio_value",
    "value_per_applicant", "pr_auc", "roc_auc"
]])

display(economic_recommendation[[
    "policy_type", "calibration_method", "split", "threshold",
    "predicted_reject_share", "approved_share", "precision_bad_rate_among_rejected",
    "recall_default_capture", "approved_bad_rate", "total_portfolio_value",
    "value_per_applicant", "recommendation_reason"
]])


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_pr_auc_ranking_with_f1_threshold_warning.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_fixed_review_volume_policy_all_available.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_top_pr_auc_fixed_review_volume_policy.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables/final_model_top_pr_auc_fixed_review_volume_missing.csv



,selection_rank,model_dataset_label,split,pr_auc,roc_auc,f1_threshold_f1,f1_threshold_precision,f1_threshold_recall,f1_threshold_predicted_reject_share,operating_policy_warning
0,1,XGBoost | missingness challenger,validation,0.425707,0.702088,0.474077,0.354607,0.714949,0.497518,Reject/share columns are from the automated be...
1,2,HistGradientBoosting | baseline no grade/subgrade,validation,0.425647,0.701741,0.474176,0.365229,0.675751,0.456564,Reject/share columns are from the automated be...
2,3,XGBoost | baseline with grade/subgrade,validation,0.425400,0.701999,0.473884,0.357988,0.700748,0.483030,Reject/share columns are from the automated be...
3,4,XGBoost | missingness challenger no grade/subg...,validation,0.425318,0.701878,0.474124,0.355525,0.711458,0.493810,Reject/share columns are from the automated be...
4,5,XGBoost | baseline no grade/subgrade,validation,0.425118,0.701651,0.474049,0.354651,0.714645,0.497245,Reject/share columns are from the automated be...
5,6,LightGBM | missingness challenger,validation,0.424804,0.702981,0.475150,0.361194,0.694157,0.474240,Reject/share columns are from the automated be...
6,7,LightGBM | baseline with grade/subgrade,validation,0.424770,0.702851,0.475468,0.354110,0.723382,0.504093,Reject/share columns are from the automated be...
7,8,LightGBM | missingness challenger no grade/sub...,validation,0.424751,0.702609,0.475249,0.355631,0.716119,0.496897,Reject/share columns are from the automated be...
8,9,LightGBM | baseline no grade/subgrade,validation,0.424132,0.702179,0.475030,0.357799,0.706515,0.487262,Reject/share columns are from the automated be...
9,10,HistGradientBoosting | missingness challenger,validation,0.423860,0.700996,0.473904,0.353878,0.717138,0.500070,Reject/share columns are from the automated be...


,selection_rank,model_dataset_label,split,review_pct,business_policy,review_count,captured_bad,precision,recall,base_bad_rate,lift_over_base_bad_rate,avoided_auto_reject_share_vs_f1_threshold
0,1,XGBoost | missingness challenger,test,5.0,review/risk-action top 5% only,9788,4880,0.498570,0.118536,0.210315,2.370583,0.447518
1,3,XGBoost | baseline with grade/subgrade,test,5.0,review/risk-action top 5% only,9788,4898,0.500409,0.118973,0.210315,2.379327,0.433030
2,4,XGBoost | missingness challenger no grade/subg...,test,5.0,review/risk-action top 5% only,9788,4864,0.496935,0.118147,0.210315,2.362810,0.443810
3,5,XGBoost | baseline no grade/subgrade,test,5.0,review/risk-action top 5% only,9788,4890,0.499591,0.118779,0.210315,2.375440,0.447245
4,6,LightGBM | missingness challenger,test,5.0,review/risk-action top 5% only,9788,4868,0.497344,0.118244,0.210315,2.364753,0.424240
5,7,LightGBM | baseline with grade/subgrade,test,5.0,review/risk-action top 5% only,9788,4879,0.498468,0.118512,0.210315,2.370097,0.454093
6,8,LightGBM | missingness challenger no grade/sub...,test,5.0,review/risk-action top 5% only,9788,4836,0.494074,0.117467,0.210315,2.349209,0.446897
7,9,LightGBM | baseline no grade/subgrade,test,5.0,review/risk-action top 5% only,9788,4829,0.493359,0.117297,0.210315,2.345808,0.437262
8,10,HistGradientBoosting | missingness challenger,test,5.0,review/risk-action top 5% only,9788,4821,0.492542,0.117103,0.210315,2.341922,0.450070
9,11,HistGradientBoosting | baseline with grade/sub...,test,5.0,review/risk-action top 5% only,9788,4831,0.493564,0.117346,0.210315,2.346780,0.428258


,selection_rank,model_dataset_label,model,dataset,missing_reason
0,2,HistGradientBoosting | baseline no grade/subgrade,hist_gradient_boosting_no_grade_subgrade,baseline_no_grade_subgrade,No fixed-review-volume table was available for...


,policy_type,calibration_method,threshold,predicted_reject_share,precision_bad_rate_among_rejected,recall_default_capture,total_portfolio_value,value_per_applicant,default_loss_saved,good_borrower_opportunity_cost,economic_break_even_probability,max_reject_share_constraint
0,economic_unconstrained,raw,0.300875,0.792997,0.289657,0.930840,271412000.0,1452.02,10000.0,1500.0,0.130435,NaN
1,economic_max_20pct_reject,raw,0.630464,0.200000,0.449952,0.364683,137365500.0,734.89,10000.0,1500.0,0.130435,0.2
2,economic_unconstrained,platt_sigmoid,0.125063,0.792997,0.289657,0.930840,271412000.0,1452.02,10000.0,1500.0,0.130435,NaN
3,economic_max_20pct_reject,platt_sigmoid,0.365439,0.200000,0.449952,0.364683,137365500.0,734.89,10000.0,1500.0,0.130435,0.2
4,economic_unconstrained,isotonic,0.130916,0.792901,0.289686,0.930818,271427500.0,1452.11,10000.0,1500.0,0.130435,NaN
5,economic_max_20pct_reject,isotonic,0.368465,0.198251,0.451116,0.362428,136660000.0,731.11,10000.0,1500.0,0.130435,0.2


,policy_type,calibration_method,split,threshold,predicted_reject_share,approved_share,precision_bad_rate_among_rejected,recall_default_capture,approved_bad_rate,total_portfolio_value,value_per_applicant,pr_auc,roc_auc
0,economic_max_20pct_reject,isotonic,test,0.368465,0.161579,0.838421,0.419267,0.322111,0.170046,105058000.0,536.70,0.375923,0.710411
1,economic_max_20pct_reject,isotonic,validation,0.368465,0.166018,0.833982,0.467163,0.314298,0.202889,120167500.0,642.88,0.422035,0.702460
2,economic_max_20pct_reject,platt_sigmoid,test,0.365439,0.194673,0.805327,0.404361,0.374286,0.163408,120043000.0,613.25,0.380189,0.710474
3,economic_max_20pct_reject,platt_sigmoid,validation,0.365439,0.200000,0.800000,0.449952,0.364683,0.195966,137365500.0,734.89,0.425707,0.702088
4,economic_max_20pct_reject,raw,test,0.630464,0.194673,0.805327,0.404361,0.374286,0.163408,120043000.0,613.25,0.380189,0.710474
5,economic_max_20pct_reject,raw,validation,0.630464,0.200000,0.800000,0.449952,0.364683,0.195966,137365500.0,734.89,0.425707,0.702088
6,economic_unconstrained,isotonic,test,0.130916,0.740213,0.259787,0.260228,0.915883,0.068098,216275000.0,1104.86,0.375923,0.710411
7,economic_unconstrained,isotonic,validation,0.130916,0.784196,0.215804,0.291448,0.926201,0.084387,271418500.0,1452.06,0.422035,0.702460
8,economic_unconstrained,platt_sigmoid,test,0.125063,0.749163,0.250837,0.258408,0.920474,0.066679,215820500.0,1102.54,0.380189,0.710474
9,economic_unconstrained,platt_sigmoid,validation,0.125063,0.792997,0.207003,0.289657,0.930840,0.082444,271412000.0,1452.02,0.425707,0.702088


,policy_type,calibration_method,split,threshold,predicted_reject_share,approved_share,precision_bad_rate_among_rejected,recall_default_capture,approved_bad_rate,total_portfolio_value,value_per_applicant,recommendation_reason
0,economic_max_20pct_reject,platt_sigmoid,test,0.365439,0.194673,0.805327,0.404361,0.374286,0.163408,120043000.0,613.25,Use calibrated probabilities and a dollar-valu...


## 10. Calibration Check

Check whether predicted default probabilities from the PR-AUC recommended ranker align with observed default rates.


In [10]:
CALIBRATION_BINS_PATH = TABLE_DIR / "final_model_calibration_bins.csv"
CALIBRATION_SUMMARY_PATH = TABLE_DIR / "final_model_calibration_summary.csv"

missing_calibration_outputs = [
    path for path in [CALIBRATION_BINS_PATH, CALIBRATION_SUMMARY_PATH]
    if not path.exists()
]
if missing_calibration_outputs:
    raise FileNotFoundError(
        "Missing calibration outputs. Run scripts/create_calibration_analysis.py first: "
        + ", ".join(str(path) for path in missing_calibration_outputs)
    )

calibration_bins = pd.read_csv(CALIBRATION_BINS_PATH)
calibration_summary = pd.read_csv(CALIBRATION_SUMMARY_PATH)

display(calibration_summary)
display(calibration_bins[[
    "calibration_method", "split", "probability_bin", "rows", "bad_count",
    "predicted_probability_mean", "observed_bad_rate", "abs_calibration_error"
]])

around_45 = calibration_bins[calibration_bins["probability_bin"] == "(0.4, 0.5]"].copy()
around_45["calibration_interpretation"] = np.where(
    around_45["calibration_method"].eq("raw"),
    "Raw probabilities near 0.45 materially overstate observed default risk; use raw scores for ranking, not probability decisions.",
    "Calibrated probabilities are closer to observed default risk and are more defensible for probability-based policy thresholds.",
)
display(around_45[[
    "calibration_method", "split", "probability_bin", "rows", "predicted_probability_mean",
    "observed_bad_rate", "abs_calibration_error", "calibration_interpretation"
]])


,model_family,model,model_dataset_label,calibration_method,split,rows,bad_rate,mean_predicted_probability,brier_score,pr_auc,roc_auc,expected_calibration_error,max_bin_abs_calibration_error
0,xgboost,xgboost_05_missingness_challenger,XGBoost | missingness challenger,isotonic,test,195749,0.210315,0.237204,0.151621,0.375923,0.710411,0.026889,0.279720
1,xgboost,xgboost_05_missingness_challenger,XGBoost | missingness challenger,isotonic,validation,186920,0.246763,0.246763,0.167857,0.422035,0.702460,0.000000,0.000001
2,xgboost,xgboost_05_missingness_challenger,XGBoost | missingness challenger,platt_sigmoid,test,195749,0.210315,0.238431,0.151708,0.380189,0.710474,0.028116,0.059208
3,xgboost,xgboost_05_missingness_challenger,XGBoost | missingness challenger,platt_sigmoid,validation,186920,0.246763,0.246768,0.168128,0.425707,0.702088,0.008255,0.050351
4,xgboost,xgboost_05_missingness_challenger,XGBoost | missingness challenger,raw,test,195749,0.210315,0.447443,0.212454,0.380189,0.710474,0.237127,0.324139
5,xgboost,xgboost_05_missingness_challenger,XGBoost | missingness challenger,raw,validation,186920,0.246763,0.463151,0.217997,0.425707,0.702088,0.216388,0.278115


,calibration_method,split,probability_bin,rows,bad_count,predicted_probability_mean,observed_bad_rate,abs_calibration_error
0,raw,validation,"(-0.001, 0.1]",3203,60,0.078366,0.018732,5.963316e-02
1,raw,validation,"(0.1, 0.2]",12554,704,0.154124,0.056078,9.804662e-02
2,raw,validation,"(0.2, 0.3]",22691,2404,0.254780,0.105945,1.488346e-01
3,raw,validation,"(0.3, 0.4]",32270,5079,0.351719,0.157391,1.943285e-01
4,raw,validation,"(0.4, 0.5]",36544,8159,0.450410,0.223265,2.271453e-01
5,raw,validation,"(0.5, 0.6]",33694,9831,0.548242,0.291773,2.564691e-01
6,raw,validation,"(0.6, 0.7]",24771,9267,0.646597,0.374107,2.724901e-01
7,raw,validation,"(0.7, 0.8]",15429,7206,0.745157,0.467043,2.781146e-01
8,raw,validation,"(0.8, 0.9]",5719,3378,0.833763,0.590663,2.431006e-01
9,raw,validation,"(0.9, 1.0]",45,37,0.905046,0.822222,8.282370e-02


,calibration_method,split,probability_bin,rows,predicted_probability_mean,observed_bad_rate,abs_calibration_error,calibration_interpretation
4,raw,validation,"(0.4, 0.5]",36544,0.450410,0.223265,2.271453e-01,Raw probabilities near 0.45 materially oversta...
14,raw,test,"(0.4, 0.5]",35193,0.450442,0.195181,2.552614e-01,Raw probabilities near 0.45 materially oversta...
24,platt_sigmoid,validation,"(0.4, 0.5]",17958,0.445823,0.427386,1.843661e-02,Calibrated probabilities are closer to observe...
34,platt_sigmoid,test,"(0.4, 0.5]",18667,0.445879,0.386672,5.920781e-02,Calibrated probabilities are closer to observe...
44,isotonic,validation,"(0.4, 0.5]",15276,0.441085,0.441084,8.381406e-07,Calibrated probabilities are closer to observe...
54,isotonic,test,"(0.4, 0.5]",15830,0.441400,0.396399,4.500036e-02,Calibrated probabilities are closer to observe...


## 11. XGBoost Class Weighting Review

Check whether `scale_pos_weight` is over-amplifying the bad class relative to a neutral XGBoost fit with the same selected hyperparameters.


In [11]:
XGBOOST_WEIGHTING_REVIEW_PATH = TABLE_DIR / "final_model_xgboost_class_weighting_review.csv"

if not XGBOOST_WEIGHTING_REVIEW_PATH.exists():
    raise FileNotFoundError(
        "Missing XGBoost class weighting review. Run scripts/review_xgboost_class_weighting.py first: "
        + str(XGBOOST_WEIGHTING_REVIEW_PATH)
    )

xgb_weighting_review = pd.read_csv(XGBOOST_WEIGHTING_REVIEW_PATH)

display(xgb_weighting_review[[
    "dataset_label", "weighting_policy", "scale_pos_weight", "split",
    "roc_auc", "pr_auc", "bad_rate", "mean_predicted_probability_raw",
    "best_f1_threshold", "best_f1_predicted_reject_share",
    "top_pct_precision", "top_pct_recall"
]])

validation_weighting_review = (
    xgb_weighting_review[xgb_weighting_review["split"] == "validation"]
    .sort_values(["pr_auc", "roc_auc"], ascending=False)
)
display(validation_weighting_review[[
    "dataset_label", "weighting_policy", "scale_pos_weight",
    "pr_auc", "roc_auc", "mean_predicted_probability_raw",
    "best_f1_predicted_reject_share", "top_pct_precision", "top_pct_recall"
]])


,dataset_label,weighting_policy,scale_pos_weight,split,roc_auc,pr_auc,bad_rate,mean_predicted_probability_raw,best_f1_threshold,best_f1_predicted_reject_share,top_pct_precision,top_pct_recall
0,missingness challenger,current_balanced_scale_pos_weight,4.302977,validation,0.702088,0.425707,0.246763,0.463151,0.464113,0.497518,0.449952,0.364683
1,missingness challenger,current_balanced_scale_pos_weight,4.302977,test,0.710474,0.380189,0.210315,0.447443,0.502559,0.405223,0.401124,0.381452
2,missingness challenger,neutral_scale_pos_weight_1,1.000000,validation,0.701743,0.425691,0.246763,0.197892,0.174170,0.476862,0.449203,0.364076
3,missingness challenger,neutral_scale_pos_weight_1,1.000000,test,0.710396,0.380674,0.210315,0.193241,0.182601,0.439532,0.400587,0.380942
4,missingness challenger no grade/subgrade,current_balanced_scale_pos_weight,4.302977,validation,0.701878,0.425318,0.246763,0.464756,0.463641,0.493816,0.449497,0.364314
5,missingness challenger no grade/subgrade,current_balanced_scale_pos_weight,4.302977,test,0.709706,0.379471,0.210315,0.452148,0.489481,0.433841,0.402146,0.382424
6,missingness challenger no grade/subgrade,neutral_scale_pos_weight_1,1.000000,validation,0.701947,0.425906,0.246763,0.200000,0.170881,0.486267,0.450193,0.364878
7,missingness challenger no grade/subgrade,neutral_scale_pos_weight_1,1.000000,test,0.710269,0.380754,0.210315,0.197448,0.195660,0.409111,0.401533,0.381841


,dataset_label,weighting_policy,scale_pos_weight,pr_auc,roc_auc,mean_predicted_probability_raw,best_f1_predicted_reject_share,top_pct_precision,top_pct_recall
6,missingness challenger no grade/subgrade,neutral_scale_pos_weight_1,1.000000,0.425906,0.701947,0.200000,0.486267,0.450193,0.364878
0,missingness challenger,current_balanced_scale_pos_weight,4.302977,0.425707,0.702088,0.463151,0.497518,0.449952,0.364683
2,missingness challenger,neutral_scale_pos_weight_1,1.000000,0.425691,0.701743,0.197892,0.476862,0.449203,0.364076
4,missingness challenger no grade/subgrade,current_balanced_scale_pos_weight,4.302977,0.425318,0.701878,0.464756,0.493816,0.449497,0.364314


## 12. Neutral XGBoost Retuning

Retune the cleaner XGBoost base model with `scale_pos_weight=1`. The standard missingness challenger is the main candidate; the no-grade/subgrade run remains an ablation check.


In [12]:
NEUTRAL_XGB_OUTPUTS = {
    "standard_missingness": {
        "label": "standard missingness challenger",
        "candidates": TABLE_DIR / "neutral_xgboost_missingness_challenger_candidate_results.csv",
        "metrics": TABLE_DIR / "neutral_xgboost_missingness_challenger_selected_model_metrics.csv",
        "calibration": TABLE_DIR / "neutral_xgboost_missingness_challenger_calibration_summary.csv",
        "policy": TABLE_DIR / "neutral_xgboost_missingness_challenger_economic_policy.csv",
    },
    "no_grade_subgrade_ablation": {
        "label": "no-grade/subgrade ablation",
        "candidates": TABLE_DIR / "neutral_xgboost_no_grade_subgrade_candidate_results.csv",
        "metrics": TABLE_DIR / "neutral_xgboost_no_grade_subgrade_selected_model_metrics.csv",
        "calibration": TABLE_DIR / "neutral_xgboost_no_grade_subgrade_calibration_summary.csv",
        "policy": TABLE_DIR / "neutral_xgboost_no_grade_subgrade_economic_policy.csv",
    },
}

missing_neutral_xgb_outputs = [
    path
    for config in NEUTRAL_XGB_OUTPUTS.values()
    for path in config.values()
    if isinstance(path, Path)
    if not path.exists()
]
if missing_neutral_xgb_outputs:
    raise FileNotFoundError(
        "Missing neutral XGBoost retuning outputs. Run scripts/tune_neutral_xgboost_missingness_challenger.py "
        "and scripts/tune_neutral_xgboost_no_grade_subgrade.py first: "
        + ", ".join(str(path) for path in missing_neutral_xgb_outputs)
    )

neutral_candidate_frames = []
neutral_metric_frames = []
neutral_calibration_frames = []
neutral_policy_frames = []
for variant, config in NEUTRAL_XGB_OUTPUTS.items():
    candidates = pd.read_csv(config["candidates"])
    metrics = pd.read_csv(config["metrics"])
    calibration = pd.read_csv(config["calibration"])
    policy = pd.read_csv(config["policy"])
    for frame in [candidates, metrics, calibration, policy]:
        frame["variant"] = variant
        frame["variant_label"] = config["label"]
    neutral_candidate_frames.append(candidates)
    neutral_metric_frames.append(metrics)
    neutral_calibration_frames.append(calibration)
    neutral_policy_frames.append(policy)

neutral_xgb_candidates = pd.concat(neutral_candidate_frames, ignore_index=True, sort=False)
neutral_xgb_metrics = pd.concat(neutral_metric_frames, ignore_index=True, sort=False)
neutral_xgb_calibration = pd.concat(neutral_calibration_frames, ignore_index=True, sort=False)
neutral_xgb_policy = pd.concat(neutral_policy_frames, ignore_index=True, sort=False)

display(neutral_xgb_candidates[[
    "variant_label", "candidate", "scale_pos_weight", "fit_seconds", "roc_auc", "pr_auc",
    "mean_predicted_probability_raw", "best_f1", "best_f1_precision",
    "best_f1_recall"
]].sort_values(["pr_auc", "roc_auc"], ascending=False).head(24))

display(neutral_xgb_metrics[[
    "variant_label", "model_dataset_label", "split", "roc_auc", "pr_auc",
    "mean_predicted_probability_raw", "best_f1_predicted_reject_share",
    "review_pct", "precision", "recall"
]])

display(neutral_xgb_calibration[[
    "variant_label", "model_dataset_label", "calibration_method", "split",
    "bad_rate", "mean_predicted_probability", "brier_score", "pr_auc", "roc_auc"
]])

display(neutral_xgb_policy[[
    "variant_label", "model_dataset_label", "policy_type", "calibration_method", "split",
    "threshold", "predicted_reject_share", "approved_share",
    "precision_bad_rate_among_rejected", "recall_default_capture",
    "approved_bad_rate", "total_portfolio_value", "value_per_applicant"
]])


,variant_label,candidate,scale_pos_weight,fit_seconds,roc_auc,pr_auc,mean_predicted_probability_raw,best_f1,best_f1_precision,best_f1_recall
12,no-grade/subgrade ablation,xgb_neutral_09,1.0,9.988,0.702038,0.426423,0.199975,0.474964,0.360500,0.695935
0,standard missingness challenger,xgb_neutral_09,1.0,15.210,0.702142,0.426323,0.197618,0.474743,0.364687,0.679935
1,standard missingness challenger,xgb_neutral_06,1.0,12.373,0.701961,0.426087,0.198053,0.474347,0.355922,0.710873
13,no-grade/subgrade ablation,xgb_neutral_06,1.0,7.596,0.701829,0.426004,0.200091,0.474727,0.360110,0.696369
14,no-grade/subgrade ablation,xgb_neutral_01,1.0,7.877,0.701947,0.425906,0.200000,0.474726,0.357816,0.705106
15,no-grade/subgrade ablation,xgb_neutral_08,1.0,11.128,0.701710,0.425854,0.199975,0.475218,0.359160,0.702092
2,standard missingness challenger,xgb_neutral_01,1.0,12.522,0.701743,0.425691,0.197892,0.474346,0.359904,0.695501
3,standard missingness challenger,xgb_neutral_08,1.0,17.739,0.701591,0.425548,0.198055,0.474700,0.360480,0.694873
16,no-grade/subgrade ablation,xgb_neutral_02,1.0,9.331,0.701069,0.425142,0.199948,0.474455,0.356694,0.708293
4,standard missingness challenger,xgb_neutral_02,1.0,15.188,0.700876,0.424886,0.198285,0.473751,0.360110,0.692184


,variant_label,model_dataset_label,split,roc_auc,pr_auc,mean_predicted_probability_raw,best_f1_predicted_reject_share,review_pct,precision,recall
0,standard missingness challenger,XGBoost neutral | missingness challenger,validation,0.702142,0.426323,0.197618,0.460074,20.0,0.449658,0.364444
1,standard missingness challenger,XGBoost neutral | missingness challenger,test,0.710495,0.380704,0.193062,0.423175,20.0,0.399796,0.380189
2,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,validation,0.702038,0.426423,0.199975,0.476370,20.0,0.449898,0.364640
3,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,test,0.710201,0.380806,0.197260,0.420666,20.0,0.400945,0.381282


,variant_label,model_dataset_label,calibration_method,split,bad_rate,mean_predicted_probability,brier_score,pr_auc,roc_auc
0,standard missingness challenger,XGBoost neutral | missingness challenger,raw,validation,0.246763,0.197618,0.170756,0.426323,0.702142
1,standard missingness challenger,XGBoost neutral | missingness challenger,raw,test,0.210315,0.193062,0.151297,0.380704,0.710495
2,standard missingness challenger,XGBoost neutral | missingness challenger,platt_sigmoid,validation,0.246763,0.246795,0.169349,0.426323,0.702142
3,standard missingness challenger,XGBoost neutral | missingness challenger,platt_sigmoid,test,0.210315,0.243746,0.153126,0.380704,0.710495
4,standard missingness challenger,XGBoost neutral | missingness challenger,isotonic,validation,0.246763,0.246764,0.167804,0.422324,0.702524
5,standard missingness challenger,XGBoost neutral | missingness challenger,isotonic,test,0.210315,0.240332,0.151835,0.375915,0.710351
6,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,raw,validation,0.246763,0.199975,0.170546,0.426423,0.702038
7,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,raw,test,0.210315,0.197260,0.151276,0.380806,0.710201
8,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,platt_sigmoid,validation,0.246763,0.246789,0.169355,0.426423,0.702038
9,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,platt_sigmoid,test,0.210315,0.245537,0.153244,0.380806,0.710201


,variant_label,model_dataset_label,policy_type,calibration_method,split,threshold,predicted_reject_share,approved_share,precision_bad_rate_among_rejected,recall_default_capture,approved_bad_rate,total_portfolio_value,value_per_applicant
0,standard missingness challenger,XGBoost neutral | missingness challenger,economic_max_20pct_reject,platt_sigmoid,validation,0.325248,0.200000,0.800000,0.449658,0.364444,0.196040,137239000.0,734.21
1,standard missingness challenger,XGBoost neutral | missingness challenger,economic_max_20pct_reject,platt_sigmoid,test,0.325248,0.201288,0.798712,0.399497,0.382351,0.162638,121918500.0,622.83
2,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,economic_max_20pct_reject,platt_sigmoid,validation,0.327936,0.200000,0.800000,0.449898,0.364640,0.195980,137342500.0,734.77
3,no-grade/subgrade ablation,XGBoost neutral | missingness challenger no gr...,economic_max_20pct_reject,platt_sigmoid,test,0.327936,0.204854,0.795146,0.399127,0.388763,0.161671,123907500.0,632.99


## 13. Grade/Subgrade and Interest Rate Ablation

Test the four requested feature combinations using neutral XGBoost: with/without `grade` and `sub_grade`, and with/without `int_rate_clean`.

Interpretation focus: `grade/sub_grade` and `int_rate_clean` carry overlapping Lending Club pricing/risk information. The preferred production-style model excludes `grade/sub_grade` and keeps `int_rate_clean` for parsimony and equivalent predictive performance.

In [13]:
GRADE_INT_FEATURE_SUMMARY_PATH = TABLE_DIR / "xgboost_grade_int_rate_feature_summary.csv"
GRADE_INT_SELECTED_PATH = TABLE_DIR / "xgboost_grade_int_rate_selected_candidates.csv"
GRADE_INT_METRICS_PATH = TABLE_DIR / "xgboost_grade_int_rate_selected_model_metrics.csv"
GRADE_INT_POLICY_PATH = TABLE_DIR / "xgboost_grade_int_rate_economic_policy.csv"

missing_grade_int_outputs = [
    path for path in [
        GRADE_INT_FEATURE_SUMMARY_PATH,
        GRADE_INT_SELECTED_PATH,
        GRADE_INT_METRICS_PATH,
        GRADE_INT_POLICY_PATH,
    ]
    if not path.exists()
]
if missing_grade_int_outputs:
    raise FileNotFoundError(
        "Missing grade/int_rate ablation outputs. Run scripts/test_xgboost_grade_int_rate_combinations.py first: "
        + ", ".join(str(path) for path in missing_grade_int_outputs)
    )

grade_int_feature_summary = pd.read_csv(GRADE_INT_FEATURE_SUMMARY_PATH)
grade_int_selected = pd.read_csv(GRADE_INT_SELECTED_PATH)
grade_int_metrics = pd.read_csv(GRADE_INT_METRICS_PATH)
grade_int_policy = pd.read_csv(GRADE_INT_POLICY_PATH)

display(grade_int_feature_summary[[
    "feature_set_label", "include_grade_subgrade", "include_int_rate",
    "kept_feature_count", "dropped_feature_count"
]])

display(grade_int_selected[[
    "selection_rank", "feature_set_label", "candidate", "feature_count",
    "pr_auc", "roc_auc", "mean_predicted_probability_raw",
    "best_f1_precision", "best_f1_recall"
]])

display(grade_int_metrics[[
    "feature_set_label", "split", "pr_auc", "roc_auc",
    "mean_predicted_probability_raw", "best_f1_predicted_reject_share",
    "review_pct", "review_precision", "review_recall"
]])

display(grade_int_policy[[
    "feature_set_label", "split", "threshold", "predicted_reject_share",
    "approved_share", "precision_bad_rate_among_rejected",
    "recall_default_capture", "approved_bad_rate",
    "total_portfolio_value", "value_per_applicant"
]])


,feature_set_label,include_grade_subgrade,include_int_rate,kept_feature_count,dropped_feature_count
0,with grade/sub_grade + int_rate,True,True,102,0
1,without grade/sub_grade + int_rate,False,True,60,42
2,with grade/sub_grade without int_rate,True,False,101,1
3,without grade/sub_grade and without int_rate,False,False,59,43


,selection_rank,feature_set_label,candidate,feature_count,pr_auc,roc_auc,mean_predicted_probability_raw,best_f1_precision,best_f1_recall
0,1,with grade/sub_grade without int_rate,xgb_neutral_01_with_grade_subgrade_without_int...,101,0.426470,0.702002,0.193563,0.351969,0.728997
1,2,without grade/sub_grade + int_rate,xgb_neutral_09_without_grade_subgrade_with_int...,60,0.426423,0.702038,0.199975,0.360500,0.695935
2,3,with grade/sub_grade + int_rate,xgb_neutral_09_with_grade_subgrade_with_int_rate,102,0.426323,0.702142,0.197618,0.364687,0.679935
3,4,without grade/sub_grade and without int_rate,xgb_neutral_06_without_grade_subgrade_without_...,59,0.422215,0.696424,0.189554,0.351418,0.708878


,feature_set_label,split,pr_auc,roc_auc,mean_predicted_probability_raw,best_f1_predicted_reject_share,review_pct,review_precision,review_recall
0,with grade/sub_grade + int_rate,validation,0.426323,0.702142,0.197618,0.460074,20.0,0.449658,0.364444
1,with grade/sub_grade + int_rate,test,0.380704,0.710495,0.193062,0.423175,20.0,0.399796,0.380189
2,without grade/sub_grade + int_rate,validation,0.426423,0.702038,0.199975,0.476370,20.0,0.449898,0.364640
3,without grade/sub_grade + int_rate,test,0.380806,0.710201,0.197260,0.420666,20.0,0.400945,0.381282
4,with grade/sub_grade without int_rate,validation,0.426470,0.702002,0.193563,0.511096,20.0,0.449497,0.364314
5,with grade/sub_grade without int_rate,test,0.381304,0.710762,0.188409,0.410536,20.0,0.399770,0.380165
6,without grade/sub_grade and without int_rate,validation,0.422215,0.696424,0.189554,0.497769,20.0,0.447544,0.362732
7,without grade/sub_grade and without int_rate,test,0.375358,0.703070,0.187451,0.433468,20.0,0.396398,0.376958


,feature_set_label,split,threshold,predicted_reject_share,approved_share,precision_bad_rate_among_rejected,recall_default_capture,approved_bad_rate,total_portfolio_value,value_per_applicant
0,with grade/sub_grade + int_rate,validation,0.325248,0.200000,0.800000,0.449658,0.364444,0.196040,137239000.0,734.21
1,with grade/sub_grade + int_rate,test,0.325248,0.201288,0.798712,0.399497,0.382351,0.162638,121918500.0,622.83
2,without grade/sub_grade + int_rate,validation,0.327936,0.200000,0.800000,0.449898,0.364640,0.195980,137342500.0,734.77
3,without grade/sub_grade + int_rate,test,0.327936,0.204854,0.795146,0.399127,0.388763,0.161671,123907500.0,632.99
4,with grade/sub_grade without int_rate,validation,0.328369,0.200000,0.800000,0.449497,0.364314,0.196080,137170000.0,733.84
5,with grade/sub_grade without int_rate,test,0.328369,0.199225,0.800775,0.399867,0.378780,0.163157,120834000.0,617.29
6,without grade/sub_grade and without int_rate,validation,0.321111,0.200000,0.800000,0.447544,0.362732,0.196568,136330500.0,729.35
7,without grade/sub_grade and without int_rate,test,0.321111,0.204859,0.795141,0.394130,0.383905,0.162957,121606000.0,621.23


### Ablation Conclusion

The ablation study shows that `grade/sub_grade` and `int_rate_clean` provide highly overlapping information. Models using either signal achieved nearly identical validation PR-AUC, test PR-AUC, reject share, and rejected bad rate. Including both did not improve performance.

Removing both produced the only noticeable decline, indicating that Lending Club's pricing/risk signal is useful, but only one of these variables is needed. Therefore, the preferred production-style model excludes `grade/sub_grade` and keeps `int_rate_clean` for parsimony and equivalent predictive performance.

## 14. Business Explanation: Recommended Model and Probability Policy

The recommended business model is **Experiment B: neutral XGBoost without `grade/sub_grade`, with `int_rate_clean`**.

This is not the absolute highest validation PR-AUC by a tiny margin. The strict statistical winner is Experiment C, which keeps `grade/sub_grade` and removes `int_rate_clean`. However, Experiment B is preferred for production-style use because it removes Lending Club's explicit risk buckets while keeping `int_rate_clean`, a continuous loan-term variable known at loan acceptance/origination.

Business interpretation:

- `grade/sub_grade` and `int_rate_clean` carry highly overlapping pricing/risk information.
- Keeping both does not improve performance.
- Removing both slightly weakens the model.
- Keeping `int_rate_clean` while removing `grade/sub_grade` preserves performance and gives a cleaner governance story.
- The model probabilities are not used directly from raw XGBoost scores; they are passed through Platt calibration before the economic underwriting policy is applied.
- The final decision policy is not the automated best-F1 threshold. It is a calibrated, capped economic policy with an approximately 20% maximum reject/review share.


In [14]:
BUSINESS_FEATURE_SET = "without_grade_subgrade_with_int_rate"

business_selected = grade_int_selected[grade_int_selected["feature_set"] == BUSINESS_FEATURE_SET].copy()
business_metrics = grade_int_metrics[grade_int_metrics["feature_set"] == BUSINESS_FEATURE_SET].copy()
business_policy = grade_int_policy[grade_int_policy["feature_set"] == BUSINESS_FEATURE_SET].copy()

business_summary = pd.DataFrame([
    {
        "business_choice": "preferred production-style model",
        "model": business_selected.iloc[0]["candidate"],
        "feature_set": business_selected.iloc[0]["feature_set_label"],
        "why": "Removes grade/sub_grade, keeps int_rate_clean, and preserves equivalent predictive performance.",
        "validation_pr_auc": business_selected.iloc[0]["pr_auc"],
        "validation_roc_auc": business_selected.iloc[0]["roc_auc"],
        "raw_mean_predicted_probability": business_selected.iloc[0]["mean_predicted_probability_raw"],
        "scale_pos_weight": business_selected.iloc[0]["scale_pos_weight"],
    }
])

display(business_summary)

display(business_metrics[[
    "feature_set_label", "split", "pr_auc", "roc_auc",
    "mean_predicted_probability_raw", "review_pct",
    "review_precision", "review_recall"
]])

display(business_policy[[
    "feature_set_label", "policy_type", "split", "threshold",
    "predicted_reject_share", "approved_share",
    "precision_bad_rate_among_rejected", "recall_default_capture",
    "approved_bad_rate", "total_portfolio_value", "value_per_applicant"
]])


,business_choice,model,feature_set,why,validation_pr_auc,validation_roc_auc,raw_mean_predicted_probability,scale_pos_weight
0,preferred production-style model,xgb_neutral_09_without_grade_subgrade_with_int...,without grade/sub_grade + int_rate,"Removes grade/sub_grade, keeps int_rate_clean,...",0.426423,0.702038,0.199975,1.0


,feature_set_label,split,pr_auc,roc_auc,mean_predicted_probability_raw,review_pct,review_precision,review_recall
2,without grade/sub_grade + int_rate,validation,0.426423,0.702038,0.199975,20.0,0.449898,0.364640
3,without grade/sub_grade + int_rate,test,0.380806,0.710201,0.197260,20.0,0.400945,0.381282


,feature_set_label,policy_type,split,threshold,predicted_reject_share,approved_share,precision_bad_rate_among_rejected,recall_default_capture,approved_bad_rate,total_portfolio_value,value_per_applicant
2,without grade/sub_grade + int_rate,platt_economic_max_20pct_reject,validation,0.327936,0.200000,0.800000,0.449898,0.364640,0.195980,137342500.0,734.77
3,without grade/sub_grade + int_rate,platt_economic_max_20pct_reject,test,0.327936,0.204854,0.795146,0.399127,0.388763,0.161671,123907500.0,632.99


### Final Business Recommendation

Use `xgb_neutral_09_without_grade_subgrade_with_int_rate` as the preferred production-style model.

The model should be interpreted as a **risk ranking and calibrated probability system**, not as an F1-threshold classifier. The recommended operating policy is:

`neutral XGBoost -> Platt-calibrated probability -> economic value threshold -> 20% max reject/review cap`

On the test set, this preferred model keeps the rejection volume near the business cap while maintaining strong separation of bad loans in the rejected group.